In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
import os


# ========== CONFIGURATION ==========
BASE_DIR = '/home/woody/iwi5/iwi5204h/HistGen/MyExperiments/REG2025_Format'


DISPLAY_NAMES = {
    'UNI1': 'UNI',
    'UNI2_4': 'UNI2'
}


def pretty_model_name(model):
    return DISPLAY_NAMES.get(model, model)


# Specify best epoch folder for each model and seed
BEST_EPOCHS = {
    'seed43': {
        'HistGen': '31',
        'TITAN': '17',
        'UNI1': '15',
        'UNI2_4': '34',
        'Conch': '11'
    },
    'seed44': {
        'HistGen': '35',
        'TITAN': '30',
        'UNI1': '27',
        'UNI2_4': '16',
        'Conch': '35'
    },
    'seed45': {
        'HistGen': '18',
        'TITAN': '40',
        'UNI1': '38',
        'UNI2_4': '38',
        'Conch': '37'
    },
    'seed46': {
        'HistGen': '35',
        'TITAN': '37',
        'UNI1': '40',
        'UNI2_4': '25',
        'Conch': '27'
    },
    'seed456789': {
        'HistGen': '39',
        'TITAN': '33',
        'UNI1': '38',
        'UNI2_4': '27',
        'Conch': '40'
    }
}


RESULTS_FILENAME = 'results.csv'


# Bootstrap configuration
N_BOOTSTRAP = 10000  # Number of bootstrap iterations
CI_LEVEL = 0.95      # Confidence level (95%)


# TITAN-first comparisons for consistent positive improvements
PAIRWISE_COMPARISONS = [
    ('TITAN', 'UNI1'),
    ('TITAN', 'UNI2_4'),
    ('TITAN', 'Conch')
]


# ========== FUNCTIONS ==========
def load_reg_score(csv_path):
    try:
        df = pd.read_csv(csv_path)
        if 'Average Ranking Score' in df.columns:
            score = df['Average Ranking Score'].iloc[0]
        else:
            score = df.iloc[0, 0]
        return float(score)
    except Exception as e:
        print(f"    ERROR loading {csv_path}: {e}")
        return None


def load_reg_scores_for_model(model, best_epochs, base_dir, results_filename):
    scores = []
    seeds = []

    for seed in sorted(best_epochs.keys()):
        if model not in best_epochs[seed]:
            print(f"  WARNING: {model} not found in {seed}")
            continue

        epoch_folder = best_epochs[seed][model]
        results_path = os.path.join(base_dir, seed, model, epoch_folder, results_filename)

        if not os.path.exists(results_path):
            print(f"  WARNING: File not found: {results_path}")
            continue

        score = load_reg_score(results_path)
        if score is not None:
            scores.append(score)
            seeds.append(seed)
            print(f"  {seed}: REG = {score:.4f}")

    return scores, seeds


def cohens_d(group1, group2):
    mean_diff = np.mean(group1) - np.mean(group2)
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0
    return mean_diff / pooled_std


def interpret_cohens_d(d):
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"


def bootstrap_ci(group1, group2, n_bootstrap=10000, ci_level=0.95):
    diffs = np.array(group1) - np.array(group2)
    n = len(diffs)
    observed_mean = np.mean(diffs)

    bootstrap_means = np.zeros(n_bootstrap)
    for i in range(n_bootstrap):
        resample_indices = np.random.choice(n, size=n, replace=True)
        resample_diffs = diffs[resample_indices]
        bootstrap_means[i] = np.mean(resample_diffs)

    alpha = 1 - ci_level
    ci_lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))

    return observed_mean, ci_lower, ci_upper


# ========== MAIN ANALYSIS ==========
def analyze_reg_scores():
    print("="*70)
    print("LOADING REG SCORES FROM ALL SEEDS")
    print("="*70)

    np.random.seed(42)

    models = list(set([model for seed_dict in BEST_EPOCHS.values() for model in seed_dict.keys()]))
    models = sorted(models)

    print(f"\nFound {len(models)} models: {models}")
    print(f"Found {len(BEST_EPOCHS)} seeds: {list(BEST_EPOCHS.keys())}")
    print(f"Bootstrap iterations: {N_BOOTSTRAP}")
    print(f"Confidence level: {CI_LEVEL*100}%")

    reg_scores = {}
    reg_seeds = {}

    for model in models:
        print(f"\n{pretty_model_name(model)}:")
        scores, seeds = load_reg_scores_for_model(model, BEST_EPOCHS, BASE_DIR, RESULTS_FILENAME)
        reg_scores[model] = scores
        reg_seeds[model] = seeds
        print(f"  Loaded {len(scores)} REG scores")
        print(f"  Mean: {np.mean(scores):.4f}, Std: {np.std(scores, ddof=1):.4f}")

    summary_rows = []
    for model in models:
        scores = reg_scores[model]
        summary_rows.append({
            'Model': pretty_model_name(model),
            'REG': f"{np.mean(scores):.4f} ± {np.std(scores, ddof=1):.4f}"
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_output = 'reg_scores_summary_formatted.csv'
    summary_df.to_csv(summary_output, index=False)
    print(f"\nModel summary saved to: {summary_output}")

    print("\n" + "="*70)
    print("STATISTICAL ANALYSIS: REG SCORES WITH BOOTSTRAP CI")
    print("="*70)

    print(f"\nPairwise comparisons with TITAN as reference model")

    results = []

    for model_a, model_b in PAIRWISE_COMPARISONS:
        print(f"\n===== {pretty_model_name(model_a)} vs {pretty_model_name(model_b)} =====")

        scores_a = reg_scores[model_a]
        scores_b = reg_scores[model_b]
        seeds_a = reg_seeds[model_a]
        seeds_b = reg_seeds[model_b]

        if len(scores_a) != len(scores_b):
            print(f"  WARNING: Different number of seeds ({len(scores_a)} vs {len(scores_b)})")
            print(f"  Skipping comparison")
            continue

        if seeds_a != seeds_b:
            print(f"  WARNING: Seed mismatch between models")
            print(f"  {pretty_model_name(model_a)} seeds: {seeds_a}")
            print(f"  {pretty_model_name(model_b)} seeds: {seeds_b}")
            print(f"  Skipping comparison")
            continue

        mean_a = np.mean(scores_a)
        std_a = np.std(scores_a, ddof=1)
        mean_b = np.mean(scores_b)
        std_b = np.std(scores_b, ddof=1)
        mean_diff = mean_a - mean_b

        print(f"  {pretty_model_name(model_a)}: {mean_a:.4f} ± {std_a:.4f}")
        print(f"  {pretty_model_name(model_b)}: {mean_b:.4f} ± {std_b:.4f}")
        print(f"  Mean difference: {mean_diff:.4f}")

        try:
            t_stat, p_value = ttest_rel(scores_a, scores_b)
            print(f"  Paired t-test: t={t_stat:.3f}, p={p_value:.4g}")
        except Exception as e:
            print(f"  ERROR in t-test: {e}")
            t_stat, p_value = np.nan, np.nan

        d = cohens_d(scores_a, scores_b)
        d_interpretation = interpret_cohens_d(d)
        print(f"  Cohen's d: {d:.3f} ({d_interpretation} effect)")

        try:
            _, ci_lower, ci_upper = bootstrap_ci(
                scores_a, scores_b,
                n_bootstrap=N_BOOTSTRAP,
                ci_level=CI_LEVEL
            )
            ci_excludes_zero = (ci_lower > 0) or (ci_upper < 0)
            print(f"  Bootstrap {int(CI_LEVEL*100)}% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
            print(f"  CI excludes zero: {ci_excludes_zero}")
        except Exception as e:
            print(f"  ERROR in bootstrap: {e}")
            ci_lower, ci_upper = np.nan, np.nan
            ci_excludes_zero = False

        is_significant = p_value < 0.05 if not np.isnan(p_value) else False
        significance_marker = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"

        print(f"  Significant: {is_significant} ({significance_marker})")

        results.append({
            'Comparison': f'{pretty_model_name(model_a)} vs {pretty_model_name(model_b)}',
            'Baseline_Mean': mean_b,
            'Baseline_Std': std_b,
            'Model_Mean': mean_a,
            'Model_Std': std_a,
            'Mean_Diff': mean_diff,
            't_statistic': t_stat,
            'p_value': p_value,
            'Cohens_d': d,
            'Effect_Size': d_interpretation,
            'Boot_CI_Lower': ci_lower,
            'Boot_CI_Upper': ci_upper,
            'CI_Excludes_Zero': ci_excludes_zero,
            'Significant': is_significant,
            'Significance': significance_marker
        })

    results_df = pd.DataFrame(results)
    output_path = 'reg_statistical_analysis_with_bootstrap.csv'
    results_df.to_csv(output_path, index=False)

    print("\n" + "="*70)
    print("ANALYSIS COMPLETE")
    print("="*70)
    print(f"Results saved to: {output_path}")

    print("\n" + "="*70)
    print("SUMMARY TABLE")
    print("="*70)

    display_cols = ['Comparison', 'Mean_Diff', 'p_value', 'Cohens_d',
                    'Boot_CI_Lower', 'Boot_CI_Upper', 'CI_Excludes_Zero', 'Significance']
    print(results_df[display_cols].to_string(index=False))

    return results_df


# ========== RUN ==========
if __name__ == "__main__":
    results_df = analyze_reg_scores()

LOADING REG SCORES FROM ALL SEEDS

Found 5 models: ['Conch', 'HistGen', 'TITAN', 'UNI1', 'UNI2_4']
Found 5 seeds: ['seed43', 'seed44', 'seed45', 'seed46', 'seed456789']
Bootstrap iterations: 10000
Confidence level: 95.0%

Conch:
  seed43: REG = 0.6917
  seed44: REG = 0.7038
  seed45: REG = 0.6607
  seed456789: REG = 0.6946
  seed46: REG = 0.7057
  Loaded 5 REG scores
  Mean: 0.6913, Std: 0.0181

HistGen:
  seed43: REG = 0.6620
  seed44: REG = 0.6602
  seed45: REG = 0.7183
  seed456789: REG = 0.6783
  seed46: REG = 0.6624
  Loaded 5 REG scores
  Mean: 0.6763, Std: 0.0246

TITAN:
  seed43: REG = 0.7750
  seed44: REG = 0.7475
  seed45: REG = 0.7167
  seed456789: REG = 0.7470
  seed46: REG = 0.7261
  Loaded 5 REG scores
  Mean: 0.7425, Std: 0.0226

UNI:
  seed43: REG = 0.7074
  seed44: REG = 0.6903
  seed45: REG = 0.6802
  seed456789: REG = 0.6746
  seed46: REG = 0.6550
  Loaded 5 REG scores
  Mean: 0.6815, Std: 0.0194

UNI2:
  seed43: REG = 0.7012
  seed44: REG = 0.7027
  seed45: REG = 0.